# Selective Rejection EDA for Cell Classifier

This notebook evaluates a trained classifier with a **reject option**:
- Predict `good` only when very confident.
- Predict `bad` only when very confident.
- Reject uncertain samples and exclude them from accepted-only metrics.

Requested protocol implemented:
- Tune thresholds on `val` only.
- Evaluate on `test` only, with fallback `test -> val -> train` (clear warning).
- Apply temperature scaling before threshold search.
- Optimize: maximize coverage subject to accepted-only recall/specificity constraints.

## 1) Configuration

In [ ]:
from pathlib import Path

# Paths
CHECKPOINT_PATH = Path("../classifier_output/clf_resnet18_freezefalse_lr0.0001_thr0.7_maskbgfalse_cv0_20260319_225955/best_model.pt")
CROPS_ROOT = Path("../classifier_output/crops")
MASK_BG_MODE = "mask_bg_false"  # mask_bg_true | mask_bg_false

# Split policy
VAL_SPLIT = "val"            # strictly used for tuning
EVAL_PREFERRED_SPLIT = "test" # evaluate on test if available
ENABLE_EVAL_FALLBACK = True    # test -> val -> train

# Calibration
APPLY_TEMPERATURE_SCALING = True

# Optimization target (on VAL, accepted-only)
TARGET_RECALL = 0.99
TARGET_SPECIFICITY = 0.965

# Baseline and search
BASELINE_THRESHOLD = 0.50
T_BAD_GRID = [round(x, 2) for x in __import__("numpy").arange(0.01, 0.50, 0.01)]
T_GOOD_GRID = [round(x, 2) for x in __import__("numpy").arange(0.51, 1.00, 0.01)]

# Runtime
BATCH_SIZE = 128
NUM_WORKERS = 4
RANDOM_SEED = 42

# Visualization samples
N_EXAMPLES_PER_PANEL = 20

## 2) Imports

In [ ]:
from __future__ import annotations

import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "cell_size").is_dir():
            return p
    raise FileNotFoundError("Could not locate repo root containing src/cell_size")


set_seed(RANDOM_SEED)
REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cell_size.classifier.dataset import IMAGENET_MEAN, IMAGENET_STD
from cell_size.classifier.models import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repo root: {REPO_ROOT}")
print(f"Device: {DEVICE}")

## 3) Helpers

In [ ]:
class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        image, target = super().__getitem__(index)
        path, _ = self.samples[index]
        return image, target, path


def resolve_checkpoint(path: Path, search_root: Path) -> Path:
    p = Path(path)
    if p.is_file():
        return p.resolve()

    candidates = sorted(
        search_root.rglob("best_model.pt"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        print(f"CHECKPOINT_PATH not found; using newest discovered checkpoint: {candidates[0]}")
        return candidates[0].resolve()

    raise FileNotFoundError(
        f"Checkpoint not found at {p}, and no best_model.pt found under {search_root}."
    )


def split_dir(base: Path, split: str) -> Path:
    d = base / split
    if not d.is_dir():
        raise FileNotFoundError(f"Split directory not found: {d}")
    if not (d / "good").is_dir() or not (d / "bad").is_dir():
        raise FileNotFoundError(f"Expected class folders good/bad in: {d}")
    return d


def resolve_eval_split(base: Path, preferred: str, fallback: bool) -> tuple[Path, str, list[str]]:
    order = [preferred]
    if fallback:
        if preferred == "test":
            order.extend(["val", "train"])
        elif preferred == "val":
            order.append("train")

    seen = set()
    order = [s for s in order if not (s in seen or seen.add(s))]

    for s in order:
        d = base / s
        if d.is_dir() and (d / "good").is_dir() and (d / "bad").is_dir():
            return d, s, order

    raise FileNotFoundError(f"No valid evaluation split found in {base}. Checked: {order}")


def build_transform(crop_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((crop_size, crop_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def infer_logits(
    model: nn.Module,
    ds: ImageFolderWithPaths,
    good_idx: int,
    batch_size: int,
    num_workers: int,
    device: torch.device,
) -> pd.DataFrame:
    loader = DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(num_workers),
        pin_memory=torch.cuda.is_available(),
    )

    rows = []
    model.eval()
    with torch.no_grad():
        for images, targets, paths in loader:
            logits = model(images.to(device)).squeeze(1).cpu().numpy()
            y_true = (targets.numpy() == good_idx).astype(int)
            p_raw = 1.0 / (1.0 + np.exp(-logits))

            for pth, yt, lg, pr in zip(paths, y_true, logits, p_raw):
                rows.append({
                    "path": str(pth),
                    "y_true": int(yt),
                    "logit": float(lg),
                    "p_raw": float(pr),
                })

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError("Inference produced no rows.")
    return out


def fit_temperature(logits: np.ndarray, y_true: np.ndarray, max_iter: int = 200) -> float:
    x = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(y_true, dtype=torch.float32)

    log_t = torch.zeros(1, requires_grad=True)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        t = torch.exp(log_t).clamp(min=1e-3, max=100.0)
        loss = criterion(x / t, y)
        loss.backward()
        return loss

    optimizer.step(closure)
    t_final = float(torch.exp(log_t).detach().cpu().item())
    return max(1e-3, min(100.0, t_final))


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "specificity": float(specificity),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def selective_predict(p_good: np.ndarray, t_bad: float, t_good: float) -> tuple[np.ndarray, np.ndarray]:
    if not (0.0 <= t_bad < t_good <= 1.0):
        raise ValueError(f"Invalid thresholds: t_bad={t_bad}, t_good={t_good}")

    accepted = (p_good <= t_bad) | (p_good >= t_good)
    y_pred = np.full(len(p_good), -1, dtype=int)  # -1 => reject
    y_pred[p_good <= t_bad] = 0
    y_pred[p_good >= t_good] = 1
    return y_pred, accepted


def evaluate_selective(y_true: np.ndarray, p_good: np.ndarray, t_bad: float, t_good: float) -> dict:
    y_pred, accepted = selective_predict(p_good, t_bad, t_good)

    n_total = len(y_true)
    n_accepted = int(accepted.sum())
    n_rejected = int(n_total - n_accepted)
    coverage = n_accepted / n_total if n_total > 0 else np.nan

    rej_pos = int(((~accepted) & (y_true == 1)).sum())
    rej_neg = int(((~accepted) & (y_true == 0)).sum())

    out = {
        "t_bad": float(t_bad),
        "t_good": float(t_good),
        "n_total": int(n_total),
        "n_accepted": int(n_accepted),
        "n_rejected": int(n_rejected),
        "coverage": float(coverage),
        "rejected_good": rej_pos,
        "rejected_bad": rej_neg,
    }

    if n_accepted == 0:
        out.update({
            "accuracy": np.nan,
            "recall": np.nan,
            "precision": np.nan,
            "f1": np.nan,
            "specificity": np.nan,
            "tn": 0,
            "fp": 0,
            "fn": 0,
            "tp": 0,
            "feasible": False,
        })
        return out

    yt = y_true[accepted]
    yp = y_pred[accepted]
    m = binary_metrics(yt, yp)
    out.update(m)
    out["feasible"] = bool(np.isfinite(m["recall"]) and np.isfinite(m["specificity"]))
    return out


def evaluate_baseline(y_true: np.ndarray, p_good: np.ndarray, threshold: float) -> dict:
    yp = (p_good >= threshold).astype(int)
    m = binary_metrics(y_true, yp)
    out = dict(m)
    out.update({
        "threshold": float(threshold),
        "coverage": 1.0,
        "n_total": int(len(y_true)),
        "n_accepted": int(len(y_true)),
        "n_rejected": 0,
        "rejected_good": 0,
        "rejected_bad": 0,
    })
    return out


def grid_search_selective(
    y_true: np.ndarray,
    p_good: np.ndarray,
    t_bad_grid: list[float],
    t_good_grid: list[float],
    min_recall: float,
    min_specificity: float,
) -> tuple[pd.DataFrame, pd.Series, str]:
    rows = []
    for t_bad in t_bad_grid:
        for t_good in t_good_grid:
            if t_bad >= t_good:
                continue
            r = evaluate_selective(y_true, p_good, t_bad, t_good)
            rows.append(r)

    sweep = pd.DataFrame(rows)

    constrained = sweep[
        (sweep["n_accepted"] > 0)
        & (sweep["recall"] >= min_recall)
        & (sweep["specificity"] >= min_specificity)
    ].copy()

    if not constrained.empty:
        best = constrained.sort_values(
            ["coverage", "f1", "specificity"], ascending=[False, False, False]
        ).iloc[0]
        return sweep, best, "constrained_optimum"

    # fallback: maximize soft objective if constraints are not jointly reachable
    eps = 1e-9
    score = (
        sweep["coverage"].fillna(0.0)
        - 3.0 * np.maximum(0.0, min_recall - sweep["recall"].fillna(0.0)) ** 2
        - 3.0 * np.maximum(0.0, min_specificity - sweep["specificity"].fillna(0.0)) ** 2
        + eps * sweep["f1"].fillna(0.0)
    )
    best_idx = score.idxmax()
    best = sweep.loc[best_idx]
    return sweep, best, "soft_penalty_fallback"


def load_rgb(path: str | Path) -> np.ndarray:
    with Image.open(path) as im:
        return np.array(im.convert("RGB"))


def plot_grid(paths: list[str], titles: list[str], title: str, ncols: int = 5):
    if len(paths) == 0:
        print(f"No examples for: {title}")
        return

    n = len(paths)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    for i in range(nrows * ncols):
        ax = axes[i // ncols, i % ncols]
        ax.axis("off")
        if i >= n:
            continue
        ax.imshow(load_rgb(paths[i]))
        ax.set_title(titles[i], fontsize=8)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

## 4) Load Model and Datasets

In [ ]:
checkpoint_path = resolve_checkpoint(Path(CHECKPOINT_PATH), REPO_ROOT / "classifier_output")
ckpt = torch.load(str(checkpoint_path), map_location=DEVICE, weights_only=False)
encoder = ckpt["encoder"]
crop_size = int(ckpt.get("crop_size", 224))

model = build_model(encoder=encoder, pretrained=False, freeze_encoder=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

base = Path(CROPS_ROOT) / MASK_BG_MODE
if not base.is_dir():
    raise FileNotFoundError(f"Mask mode folder not found: {base}")

val_dir = split_dir(base, VAL_SPLIT)  # strict val-only tuning per request

eval_dir, eval_split, eval_checked = resolve_eval_split(
    base=base,
    preferred=EVAL_PREFERRED_SPLIT,
    fallback=ENABLE_EVAL_FALLBACK,
)

print(f"Checkpoint: {checkpoint_path}")
print(f"Encoder: {encoder}")
print(f"Crop size: {crop_size}")
print(f"Val split dir: {val_dir}")
print(f"Eval requested: {EVAL_PREFERRED_SPLIT}")
print(f"Eval checked order: {eval_checked}")
print(f"Eval resolved split: {eval_split}")
print(f"Eval dir: {eval_dir}")

if eval_split != "test":
    warnings.warn(
        f"Evaluation is not on test (resolved to '{eval_split}'). "
        "Interpret final numbers as less reliable due to fallback.",
        stacklevel=2,
    )

transform = build_transform(crop_size)
val_ds = ImageFolderWithPaths(str(val_dir), transform=transform)
eval_ds = ImageFolderWithPaths(str(eval_dir), transform=transform)

if "good" not in val_ds.class_to_idx or "bad" not in val_ds.class_to_idx:
    raise ValueError(f"VAL split missing good/bad classes: {val_ds.class_to_idx}")
if val_ds.class_to_idx != eval_ds.class_to_idx:
    raise ValueError(
        f"Class index mismatch between val and eval: {val_ds.class_to_idx} vs {eval_ds.class_to_idx}"
    )

good_idx = val_ds.class_to_idx["good"]
print(f"class_to_idx: {val_ds.class_to_idx}")
print(f"VAL samples: {len(val_ds)} | EVAL samples: {len(eval_ds)}")

## 5) Run Fresh Inference (VAL + EVAL)

In [ ]:
val_df = infer_logits(model, val_ds, good_idx, BATCH_SIZE, NUM_WORKERS, DEVICE)
eval_df = infer_logits(model, eval_ds, good_idx, BATCH_SIZE, NUM_WORKERS, DEVICE)

print(f"VAL rows: {len(val_df)}")
print(f"EVAL rows: {len(eval_df)}")

val_df.head()

## 6) Temperature Scaling (fit on VAL)

In [ ]:
if APPLY_TEMPERATURE_SCALING:
    temp = fit_temperature(
        logits=val_df["logit"].to_numpy(dtype=float),
        y_true=val_df["y_true"].to_numpy(dtype=int),
    )
else:
    temp = 1.0

val_df["p_good"] = sigmoid_np(val_df["logit"].to_numpy(dtype=float) / temp)
eval_df["p_good"] = sigmoid_np(eval_df["logit"].to_numpy(dtype=float) / temp)

print(f"Temperature used: {temp:.4f}")

## 7) Threshold Search on VAL (Selective Policy)

Objective: maximize coverage subject to:
- accepted-only recall >= TARGET_RECALL
- accepted-only specificity >= TARGET_SPECIFICITY

In [ ]:
val_y = val_df["y_true"].to_numpy(dtype=int)
val_p = val_df["p_good"].to_numpy(dtype=float)

sweep_df, best_row, search_mode = grid_search_selective(
    y_true=val_y,
    p_good=val_p,
    t_bad_grid=list(T_BAD_GRID),
    t_good_grid=list(T_GOOD_GRID),
    min_recall=float(TARGET_RECALL),
    min_specificity=float(TARGET_SPECIFICITY),
)

best_t_bad = float(best_row["t_bad"])
best_t_good = float(best_row["t_good"])

print(f"Search mode: {search_mode}")
print(f"Selected thresholds: t_bad={best_t_bad:.2f}, t_good={best_t_good:.2f}")
print(
    "Selected VAL metrics: "
    f"coverage={best_row['coverage']:.4f}, recall={best_row['recall']:.4f}, "
    f"specificity={best_row['specificity']:.4f}, f1={best_row['f1']:.4f}"
)

constrained_count = int(((sweep_df["recall"] >= TARGET_RECALL) & (sweep_df["specificity"] >= TARGET_SPECIFICITY)).sum())
print(f"Constraint-satisfying threshold pairs on VAL: {constrained_count}")

## 8) Evaluate on EVAL Split: Baseline vs Selective

In [ ]:
eval_y = eval_df["y_true"].to_numpy(dtype=int)
eval_p = eval_df["p_good"].to_numpy(dtype=float)

baseline = evaluate_baseline(eval_y, eval_p, BASELINE_THRESHOLD)
selective = evaluate_selective(eval_y, eval_p, best_t_bad, best_t_good)

compare_df = pd.DataFrame([
    {
        "policy": "baseline_single_threshold",
        "threshold_info": f"t={BASELINE_THRESHOLD:.2f}",
        "coverage": baseline["coverage"],
        "accuracy": baseline["accuracy"],
        "recall": baseline["recall"],
        "precision": baseline["precision"],
        "f1": baseline["f1"],
        "specificity": baseline["specificity"],
        "n_accepted": baseline["n_accepted"],
        "n_rejected": baseline["n_rejected"],
        "fp": baseline["fp"],
        "fn": baseline["fn"],
    },
    {
        "policy": "selective_reject",
        "threshold_info": f"t_bad={best_t_bad:.2f}, t_good={best_t_good:.2f}",
        "coverage": selective["coverage"],
        "accuracy": selective["accuracy"],
        "recall": selective["recall"],
        "precision": selective["precision"],
        "f1": selective["f1"],
        "specificity": selective["specificity"],
        "n_accepted": selective["n_accepted"],
        "n_rejected": selective["n_rejected"],
        "fp": selective["fp"],
        "fn": selective["fn"],
    },
])

display(compare_df.round(4))

## 9) FP Rejection Impact and Confusion Matrices

In [ ]:
# Baseline predictions
baseline_pred = (eval_p >= BASELINE_THRESHOLD).astype(int)

# Selective predictions
sel_pred, sel_acc = selective_predict(eval_p, best_t_bad, best_t_good)

# How many baseline FPs got rejected
baseline_fp = (eval_y == 0) & (baseline_pred == 1)
rejected_fp = baseline_fp & (~sel_acc)

print(f"Baseline FP count: {int(baseline_fp.sum())}")
print(f"Baseline FP rejected by selective policy: {int(rejected_fp.sum())}")

# Confusion matrices
cm_base = confusion_matrix(eval_y, baseline_pred, labels=[0, 1])

if sel_acc.sum() > 0:
    cm_sel = confusion_matrix(eval_y[sel_acc], sel_pred[sel_acc], labels=[0, 1])
else:
    cm_sel = np.zeros((2, 2), dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, title in [
    (axes[0], cm_base, "Baseline (All Samples)"),
    (axes[1], cm_sel, "Selective (Accepted Samples Only)"),
]:
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1], labels=["bad", "good"])
    ax.set_yticks([0, 1], labels=["bad", "good"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(
    f"Rejected composition: good={selective['rejected_good']}, "
    f"bad={selective['rejected_bad']}"
)

## 10) Search Landscape Visualizations (VAL)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Coverage vs recall/specificity scatter
axes[0].scatter(sweep_df["coverage"], sweep_df["recall"], s=10, alpha=0.25, label="recall")
axes[0].scatter(sweep_df["coverage"], sweep_df["specificity"], s=10, alpha=0.25, label="specificity")
axes[0].axhline(TARGET_RECALL, color="tab:blue", linestyle="--", linewidth=1)
axes[0].axhline(TARGET_SPECIFICITY, color="tab:orange", linestyle="--", linewidth=1)
axes[0].set_xlabel("Coverage")
axes[0].set_ylabel("Metric")
axes[0].set_title("VAL Search Cloud")
axes[0].legend(loc="best")
axes[0].grid(alpha=0.25)

# Heatmap of coverage for constraint-satisfying pairs
feasible = sweep_df[
    (sweep_df["recall"] >= TARGET_RECALL) & (sweep_df["specificity"] >= TARGET_SPECIFICITY)
].copy()

if feasible.empty:
    axes[1].text(0.5, 0.5, "No feasible threshold pairs", ha="center", va="center")
    axes[1].set_axis_off()
else:
    pivot = feasible.pivot_table(index="t_bad", columns="t_good", values="coverage", aggfunc="max")
    mat = pivot.to_numpy()
    im = axes[1].imshow(mat, aspect="auto", origin="lower", cmap="viridis")
    axes[1].set_title("Coverage Heatmap (Feasible VAL Pairs)")
    axes[1].set_xlabel("t_good index")
    axes[1].set_ylabel("t_bad index")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 11) Probability Distribution and Reject Band (EVAL)

In [ ]:
p_bad = eval_df.loc[eval_df["y_true"] == 0, "p_good"].to_numpy()
p_good = eval_df.loc[eval_df["y_true"] == 1, "p_good"].to_numpy()

plt.figure(figsize=(10, 4))
plt.hist(p_bad, bins=40, alpha=0.65, label="true bad", color="tab:orange")
plt.hist(p_good, bins=40, alpha=0.65, label="true good", color="tab:green")
plt.axvline(best_t_bad, color="red", linestyle="--", linewidth=1.5, label=f"t_bad={best_t_bad:.2f}")
plt.axvline(best_t_good, color="blue", linestyle="--", linewidth=1.5, label=f"t_good={best_t_good:.2f}")
plt.axvspan(best_t_bad, best_t_good, color="gray", alpha=0.2, label="reject zone")
plt.xlabel("P(good)")
plt.ylabel("Count")
plt.title("EVAL Probability Distribution with Reject Zone")
plt.legend(loc="best")
plt.grid(alpha=0.25)
plt.show()

## 12) Qualitative Panels: Sure TP, Sure TN, Rejected

In [ ]:
view_df = eval_df.copy()
view_df["y_pred_sel"], view_df["accepted"] = selective_predict(
    view_df["p_good"].to_numpy(), best_t_bad, best_t_good
)

view_df["certainty_good"] = view_df["p_good"]
view_df["certainty_bad"] = 1.0 - view_df["p_good"]
view_df["uncertainty"] = 1.0 - np.abs(2.0 * view_df["p_good"] - 1.0)

sure_tp = view_df[(view_df["accepted"]) & (view_df["y_true"] == 1) & (view_df["y_pred_sel"] == 1)].sort_values("certainty_good", ascending=False)
sure_tn = view_df[(view_df["accepted"]) & (view_df["y_true"] == 0) & (view_df["y_pred_sel"] == 0)].sort_values("certainty_bad", ascending=False)
rejected = view_df[(~view_df["accepted"])].sort_values("uncertainty", ascending=False)

for name, sub, score_col in [
    ("Sure TP (accepted good)", sure_tp, "p_good"),
    ("Sure TN (accepted bad)", sure_tn, "p_good"),
    ("Rejected uncertain cases", rejected, "p_good"),
]:
    pick = sub.head(int(N_EXAMPLES_PER_PANEL)).copy()
    paths = pick["path"].tolist()
    titles = [
        f"true={'good' if int(r.y_true)==1 else 'bad'} p={getattr(r, score_col):.2f}" for r in pick.itertuples(index=False)
    ]
    plot_grid(paths, titles, name, ncols=5)

## 13) Plot False Positives and False Negatives


In [ ]:
# Accepted-only mistakes under selective policy
fp_acc = view_df[(view_df["accepted"]) & (view_df["y_true"] == 0) & (view_df["y_pred_sel"] == 1)].copy()
fn_acc = view_df[(view_df["accepted"]) & (view_df["y_true"] == 1) & (view_df["y_pred_sel"] == 0)].copy()


def accepted_margin_to_boundary(p: float, t_bad: float, t_good: float) -> float:
    """Distance to nearest selective-accept boundary (smaller => harder case)."""
    if p <= t_bad:
        return float(t_bad - p)
    if p >= t_good:
        return float(p - t_good)
    return float(np.nan)


def add_hardness_columns(df: pd.DataFrame, t_bad: float, t_good: float) -> pd.DataFrame:
    out = df.copy()
    out["boundary_margin"] = out["p_good"].apply(lambda x: accepted_margin_to_boundary(float(x), t_bad, t_good))
    return out


def pred_label_from_binary(y_pred: int) -> str:
    return "good" if int(y_pred) == 1 else "bad"


def plot_hard_cases(df: pd.DataFrame, title: str, n_examples: int, seed: int) -> None:
    if df.empty:
        print(f"No accepted cases for: {title}")
        return

    # Hardest errors first: closest to acceptance boundary, then higher uncertainty.
    # Mergesort keeps deterministic ordering for ties.
    ordered = df.sort_values(
        ["boundary_margin", "uncertainty", "path"],
        ascending=[True, False, True],
        kind="mergesort",
    ).copy()

    # Deterministic tie-break shuffle for exact-equal rows (rare, but controlled).
    rng = np.random.default_rng(int(seed))
    if len(ordered) > 1:
        ordered["_tie_rand"] = rng.random(len(ordered))
        ordered = ordered.sort_values(
            ["boundary_margin", "uncertainty", "_tie_rand", "path"],
            ascending=[True, False, True, True],
            kind="mergesort",
        )

    pick = ordered.head(int(n_examples)).copy()

    paths = pick["path"].tolist()
    titles = []
    for r in pick.itertuples(index=False):
        true_lbl = "good" if int(r.y_true) == 1 else "bad"
        pred_lbl = pred_label_from_binary(int(r.y_pred_sel))
        titles.append(
            f"true={true_lbl} pred={pred_lbl} p={float(r.p_good):.3f} "
            f"unc={float(r.uncertainty):.3f} m={float(r.boundary_margin):.3f}"
        )

    plot_grid(paths, titles, title, ncols=5)


fp_acc = add_hardness_columns(fp_acc, best_t_bad, best_t_good)
fn_acc = add_hardness_columns(fn_acc, best_t_bad, best_t_good)

print(f"Accepted FP count: {len(fp_acc)}")
print(f"Accepted FN count: {len(fn_acc)}")

plot_hard_cases(
    fp_acc,
    title=f"Accepted False Positives (Hardest Near t_good={best_t_good:.2f})",
    n_examples=int(N_EXAMPLES_PER_PANEL),
    seed=int(RANDOM_SEED),
)
plot_hard_cases(
    fn_acc,
    title=f"Accepted False Negatives (Hardest Near t_bad={best_t_bad:.2f})",
    n_examples=int(N_EXAMPLES_PER_PANEL),
    seed=int(RANDOM_SEED) + 1,
)



## 14) Final Summary


In [ ]:
meets_val_constraints = bool(
    (best_row["recall"] >= TARGET_RECALL) and (best_row["specificity"] >= TARGET_SPECIFICITY)
)

print("Selective policy summary")
print("-" * 60)
print(f"Calibration: {'on' if APPLY_TEMPERATURE_SCALING else 'off'} (temperature={temp:.4f})")
print(f"Thresholds from VAL: t_bad={best_t_bad:.2f}, t_good={best_t_good:.2f}")
print(f"VAL optimization mode: {search_mode}")
print(f"VAL constraints met: {meets_val_constraints}")
print()
print("EVAL outcome")
print(f"- split used: {eval_split}")
print(f"- coverage: {selective['coverage']:.4f}")
print(f"- recall (accepted-only): {selective['recall']:.4f}")
print(f"- specificity (accepted-only): {selective['specificity']:.4f}")
print(f"- f1 (accepted-only): {selective['f1']:.4f}")
print(f"- rejected: {selective['n_rejected']} / {selective['n_total']}")
print(f"- rejected good: {selective['rejected_good']}, rejected bad: {selective['rejected_bad']}")

if eval_split != 'test':
    print("WARNING: evaluation fallback was used; numbers are less reliable than test-only evaluation.")